# Simplified Pipeline — PM2.5 in Kampala and Nairobi

**By the end of this notebook, you will be able to:**
- Train a scikit-learn model using the `fit`/`predict` pattern shared by (almost) every model in the library
- Evaluate a model by testing it on a city it never trained on, instead of a random split
- Explain, concretely, why a raw geographic coordinate can break a linear model

Goal: build, end to end, the smallest complete and methodologically correct Machine Learning pipeline: load, clean, split, engineer features, train a model, evaluate. Each step is a block of code to complete. The train/test split principle here is deliberately simple: **train on one city, test on the other**.

In [108]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TRAIN_CITY = "Nairobi" #"Kampala"
TEST_CITY = "Kampala" #"Nairobi"
COLUMNS = [
    "city", "date", "hour", "site_latitude", "site_longitude", "pm2_5",
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]

## Step 1 — Load and restrict the scope

Load `data/train.csv`, keep only the two cities and the columns listed above. Then, so the pipeline stays fast to run and easy to inspect, cap the sample at 1200 rows per city.

**Documentation references:**
- [`Series.isin()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)
- [`DataFrame.sample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)

**Exercise:** Filter the raw data to the two cities and the listed columns. Then, city by city, draw a random sample of at most 1200 rows — pass a fixed `random_state` so the sample, and every metric computed from it later, stays reproducible between runs. Combine the two samples back into one table.

In [109]:
# TODO 1: load train.csv, filter the two cities and the columns, cap at 1200 rows/city
train_data_raw = pd.read_csv("../../data/train.csv")
train_data_right_columns = train_data_raw[COLUMNS]

cities = [TRAIN_CITY, TEST_CITY]

train_data_right_columns_and_cities = train_data_right_columns[train_data_right_columns["city"].isin(cities)]
train_data_right_columns_and_cities

samples = []
for city in cities:
    train_data_city = train_data_right_columns_and_cities[train_data_right_columns_and_cities["city"]==city]
    samples.append(train_data_city.sample(n = min(len(train_data_city), 1200), random_state = 67))

train_data = pd.concat(samples, ignore_index=True)
train_data

,city,date,hour,site_latitude,site_longitude,pm2_5,sulphurdioxide_so2_column_number_density,carbonmonoxide_co_column_number_density,nitrogendioxide_no2_column_number_density,formaldehyde_tropospheric_hcho_column_number_density,uvaerosolindex_absorbing_aerosol_index,ozone_o3_column_number_density,uvaerosollayerheight_aerosol_height,cloud_cloud_fraction
0,Nairobi,2023-07-11,11,-1.239630,36.793760,7.797143,0.000420,0.024192,0.000076,0.000295,-0.467822,0.117312,NaN,0.071506
1,Nairobi,2023-11-01,11,-1.224292,36.805005,12.463163,NaN,NaN,NaN,NaN,0.244480,0.114330,NaN,NaN
2,Nairobi,2023-10-05,11,-1.286301,36.830761,13.729615,NaN,NaN,NaN,NaN,-0.517393,0.118374,NaN,NaN
3,Nairobi,2024-01-14,11,-1.243396,36.886487,17.103889,0.000144,0.033678,0.000067,0.000125,-0.551018,0.106735,NaN,NaN
4,Nairobi,2023-11-25,10,-1.283724,36.810364,10.085625,-0.000283,0.023627,0.000046,0.000256,-1.040706,0.114519,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2395,Kampala,2023-03-10,11,0.405320,32.620270,19.624400,NaN,NaN,NaN,NaN,-0.700885,0.121777,NaN,1.000000
2396,Kampala,2023-03-06,11,0.347014,32.649360,18.509600,0.000286,0.036683,0.000049,0.000203,-0.108944,0.124607,2656.096118,NaN
2397,Kampala,2023-04-26,11,0.381576,32.647109,12.934500,NaN,NaN,0.000048,-0.000102,-0.343024,0.120656,NaN,0.405135
2398,Kampala,2023-04-03,10,0.374653,32.557440,22.883800,NaN,0.032927,NaN,0.000229,-0.988339,0.121734,NaN,0.328528


## Step 2 — Clean missing values

Some satellite measurements are missing for some passes — you already measured how much, per city, in the discovery notebook. Fill them **within each city**, sorted by date: never let one city's values fill another city's gaps, that would be a geographic information leak.

**Documentation references:**
- [`DataFrameGroupBy.transform()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.transform.html)
- [`Series.ffill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.ffill.html) / [`Series.bfill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.bfill.html)

**Exercise:** Sort the rows by city and date. Then, for each fillable column, group by city and use `transform` to apply forward-fill followed by backward-fill *within* each group — `transform` is what lets you get a filled column back in the original row order, ready to reassign to `df`, rather than one small table per city to reassemble yourself.

In [110]:
# TODO 2: fill missing values by city, sorted by date
train_data["date"] = pd.to_datetime(train_data["date"])
missing_value_by_city_by_date = (train_data.groupby(["city", "date"]).apply(lambda group: group.isna().mean()).sort_index())
train_data = train_data.sort_values(["city", "date"])
print(train_data)
incomplete_columns = [colonne for colonne in COLUMNS if colonne not in ["city","date"]]
train_data[incomplete_columns] = train_data.groupby("city")[incomplete_columns].transform(
    lambda group: group.ffill().bfill()
)

missing_value_by_city_by_date = (train_data.groupby(["city", "date"]).apply(lambda group: group.isna().mean()).sort_index())
missing_value_by_city_by_date

         city       date  hour  site_latitude  site_longitude      pm2_5  \
1270  Kampala 2023-01-02    10       0.381576       32.647109  18.386100   
1481  Kampala 2023-01-02    10       0.308520       32.592240  12.529800   
1538  Kampala 2023-01-02    10       0.292820       32.571320  20.839100   
1952  Kampala 2023-01-02    10       0.347014       32.649360  17.765300   
2033  Kampala 2023-01-02    10       0.374653       32.557440  17.506000   
...       ...        ...   ...            ...             ...        ...   
373   Nairobi 2024-01-27    10      -1.283724       36.810364  13.334286   
465   Nairobi 2024-01-27    10      -1.268877       36.819139  13.103000   
677   Nairobi 2024-01-27    10      -1.243396       36.886487  12.566731   
830   Nairobi 2024-01-27    10      -1.311080       36.861865  11.920556   
1176  Nairobi 2024-01-27    10      -1.287094       36.821671  17.412500   

      sulphurdioxide_so2_column_number_density  \
1270                                 

/var/folders/7d/mtnr2ts91fj4fz94jfr927880000gn/T/ipykernel_94738/3617745049.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_value_by_city_by_date = (train_data.groupby(["city", "date"]).apply(lambda group: group.isna().mean()).sort_index())
/var/folders/7d/mtnr2ts91fj4fz94jfr927880000gn/T/ipykernel_94738/3617745049.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_value_by_city_by_date = (train

city  date  hour  site_latitude  site_longitude  pm2_5  \
city    date                                                                 
Kampala 2023-01-02   0.0   0.0   0.0            0.0             0.0    0.0   
        2023-01-03   0.0   0.0   0.0            0.0             0.0    0.0   
        2023-01-04   0.0   0.0   0.0            0.0             0.0    0.0   
        2023-01-05   0.0   0.0   0.0            0.0             0.0    0.0   
        2023-01-06   0.0   0.0   0.0            0.0             0.0    0.0   
...                  ...   ...   ...            ...             ...    ...   
Nairobi 2024-01-17   0.0   0.0   0.0            0.0             0.0    0.0   
        2024-01-24   0.0   0.0   0.0            0.0             0.0    0.0   
        2024-01-25   0.0   0.0   0.0            0.0             0.0    0.0   
        2024-01-26   0.0   0.0   0.0            0.0             0.0    0.0   
        2024-01-27   0.0   0.0   0.0            0.0             0.0    0.0   

                    sulphurdioxide_so2_column_number_density  \
city    date                                                   
Kampala 2023-01-02                                       0.0   
        2023-01-03                                       0.0   
        2023-01-04                                       0.0   
        2023-01-05                                       0.0   
        2023-01-06                                       0.0   
...                                                      ...   
Nairobi 2024-01-17                                       0.0   
        2024-01-24                                       0.0   
        2024-01-25                                       0.0   
        2024-01-26                                       0.0   
        2024-01-27                                       0.0   

                    carbonmonoxide_co_column_number_density  \
city    date                                                  
Kampala 2023-01-02                                      0.0   
        2023-01-03                                      0.0   
        2023-01-04                                      0.0   
        2023-01-05                                      0.0   
        2023-01-06                                      0.0   
...                                                     ...   
Nairobi 2024-01-17                                      0.0   
        2024-01-24                                      0.0   
        2024-01-25                                      0.0   
        2024-01-26                                      0.0   
        2024-01-27                                      0.0   

                    nitrogendioxide_no2_column_number_density  \
city    date                                                    
Kampala 2023-01-02                                        0.0   
        2023-01-03                                        0.0   
        2023-01-04                                        0.0   
        2023-01-05                                        0.0   
        2023-01-06                                        0.0   
...                                                       ...   
Nairobi 2024-01-17                                        0.0   
        2024-01-24                                        0.0   
        2024-01-25                                        0.0   
        2024-01-26                                        0.0   
        2024-01-27                                        0.0   

                    formaldehyde_tropospheric_hcho_column_number_density  \
city    date                                                               
Kampala 2023-01-02                                                0.0      
        2023-01-03                                                0.0      
        2023-01-04                                                0.0      
        2023-01-05                                                0.0      
        2023-01-06                                

## Step 3 — Manual split by city

A basic way to check whether a model actually generalizes is to test it on data it has never seen. Here, the simplest version of that idea: train on `TRAIN_CITY`, evaluate on `TEST_CITY` — no random shuffling of rows. That is enough to get started in this activity. Later on, you will see a more rigorous and systematic way to do this across several groups at once, and to guard against overfitting — cross-validation, covered in a dedicated Session 3 module.

**Exercise:** Split `df` into `train_df` (rows where `city == TRAIN_CITY`) and `test_df` (rows where `city == TEST_CITY`).

In [111]:
# TODO 3: split into train_df (TRAIN_CITY) and test_df (TEST_CITY)
train_df = train_data[train_data["city"] == TRAIN_CITY].copy()
test_df = train_data[train_data["city"] == TEST_CITY].copy()
test_df

,city,date,hour,site_latitude,site_longitude,pm2_5,sulphurdioxide_so2_column_number_density,carbonmonoxide_co_column_number_density,nitrogendioxide_no2_column_number_density,formaldehyde_tropospheric_hcho_column_number_density,uvaerosolindex_absorbing_aerosol_index,ozone_o3_column_number_density,uvaerosollayerheight_aerosol_height,cloud_cloud_fraction
1270,Kampala,2023-01-02,10,0.381576,32.647109,18.3861,0.000598,0.031420,0.000065,0.000177,-0.708565,0.119710,1223.600959,0.045968
1481,Kampala,2023-01-02,10,0.308520,32.592240,12.5298,0.000182,0.034352,0.000070,0.000214,-0.466462,0.116887,1223.600959,0.045968
1538,Kampala,2023-01-02,10,0.292820,32.571320,20.8391,0.000182,0.034518,0.000070,0.000214,-0.466462,0.116887,1223.600959,0.045968
1952,Kampala,2023-01-02,10,0.347014,32.649360,17.7653,-0.000343,0.031420,0.000043,0.000117,-0.799090,0.118242,1223.600959,0.045968
2033,Kampala,2023-01-02,10,0.374653,32.557440,17.5060,0.000215,0.031420,0.000056,0.000243,-0.489162,0.119604,1223.600959,0.045968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,Kampala,2024-02-26,11,0.380684,32.576510,22.9718,0.000535,0.036943,0.000083,0.000238,-0.374066,0.107214,3120.205558,1.000000
1611,Kampala,2024-02-26,11,0.308520,32.592240,30.3999,0.000327,0.045686,0.000080,0.000265,-0.287240,0.108288,3377.068115,1.000000
1631,Kampala,2024-02-26,11,0.344000,32.553000,43.2961,-0.000108,0.045518,0.000066,0.000268,-0.180978,0.105137,2867.584748,1.000000
1742,Kampala,2024-02-26,11,0.347014,32.649360,21.5411,-0.000044,0.042319,0.000071,0.000304,-0.427868,0.106833,2867.584748,1.000000


## Step 4 — Temporal features

A model can only exploit the patterns it can actually see in its input. A raw date doesn't expose seasonality or a weekly rhythm — nothing in a linear model lets it notice "this is a rainy month" or "this is a weekday" just by looking at a timestamp. When you know, from domain knowledge, that a pattern is likely to matter — here, pollution following seasonal and weekly cycles from traffic and economic activity — it is worth engineering it explicitly as a feature rather than hoping the model finds it on its own. That is why the next step extracts the month and the day of week from `date`.

**Exercise:** For both `train_df` and `test_df`, add `month` and `dayofweek` columns from `date`, using the same `.dt` accessor you already used in the discovery notebook.

In [112]:
# TODO 4: add month and dayofweek columns derived from date
for data in [train_df, test_df]:
    data["month"] = data["date"].dt.month
    data["dayofweek"] = data["date"].dt.dayofweek
train_df

,city,date,hour,site_latitude,site_longitude,pm2_5,sulphurdioxide_so2_column_number_density,carbonmonoxide_co_column_number_density,nitrogendioxide_no2_column_number_density,formaldehyde_tropospheric_hcho_column_number_density,uvaerosolindex_absorbing_aerosol_index,ozone_o3_column_number_density,uvaerosollayerheight_aerosol_height,cloud_cloud_fraction,month,dayofweek
48,Nairobi,2023-05-01,10,-1.283724,36.810364,14.471429,-0.000149,0.022183,0.000059,0.000037,-1.248306,0.119112,2214.041463,0.689741,5,0
181,Nairobi,2023-05-01,10,-1.286301,36.830761,25.070000,-0.000149,0.022183,0.000059,0.000037,-1.224046,0.119405,2214.041463,0.689741,5,0
194,Nairobi,2023-05-01,10,-1.287094,36.821671,8.583824,-0.000149,0.022183,0.000059,0.000037,-1.239599,0.119217,2214.041463,0.689741,5,0
299,Nairobi,2023-05-01,10,-1.311080,36.861865,6.905000,-0.000149,0.022183,0.000059,0.000037,-1.208010,0.119598,2214.041463,0.689741,5,0
336,Nairobi,2023-05-01,10,-1.224292,36.805005,13.995632,-0.000149,0.022183,0.000059,0.000037,-1.210514,0.119874,2214.041463,0.689741,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,Nairobi,2024-01-27,10,-1.283724,36.810364,13.334286,0.000199,0.029279,0.000069,0.000270,-0.970830,0.110373,3524.287842,0.047826,1,5
465,Nairobi,2024-01-27,10,-1.268877,36.819139,13.103000,0.000089,0.029279,0.000063,0.000232,-0.927529,0.110096,3524.287842,0.047826,1,5
677,Nairobi,2024-01-27,10,-1.243396,36.886487,12.566731,0.000204,0.029279,0.000038,0.000260,-0.817342,0.108149,3524.287842,0.166835,1,5
830,Nairobi,2024-01-27,10,-1.311080,36.861865,11.920556,0.000403,0.028765,0.000039,0.000209,-0.659846,0.108158,3524.287842,0.156957,1,5


## Step 5 — Baseline model and evaluation

Every scikit-learn model follows the same two-step contract, no matter how simple or advanced: `model.fit(X_train, y_train)` learns from data, `model.predict(X_test)` applies what it learned to new data. You will reuse this exact pattern for every model in this course, starting with the simplest one, `LinearRegression`.

**Warning**: `site_latitude`/`site_longitude` are deliberately excluded from the features. In the training sample, one city only occupies a small geographic area (a handful of very close monitoring sites): a linear model fits a huge coefficient on these coordinates, which explodes as soon as it is applied to very different coordinates (the other city). Try adding them back to `feature_cols` to see the effect on the metrics.

For a refresher on what RMSE, MAE, and R² actually measure, see [section 6.2 of Machine Learning Techniques: From Theory to Practice](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#62-evaluation-metrics-for-regression).

**Documentation references:**
- [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [`mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [`mean_absolute_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), [`r2_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

**Exercise:** Build `feature_cols`: every numeric column of `train_df` except `pm2_5` (the target) and the columns named in the warning above. Fit a `LinearRegression` on `train_df`, predict on `test_df`, then compute RMSE (the square root of `mean_squared_error`), MAE, and R².

In [113]:
# TODO 5: define the features, train the linear model, compute RMSE/MAE/R2
feature_cols = ( train_df.select_dtypes(include="number").columns.drop(["pm2_5", "site_latitude", "site_longitude"], errors="ignore").tolist())
X_train = train_df[feature_cols]
Y_train = train_df["pm2_5"]

X_test = test_df[feature_cols]
Y_test = test_df["pm2_5"]

model = LinearRegression(fit_intercept = True, copy_X = True)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)


rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R²:   {r2:.3f}")


RMSE: 15.249
MAE:  10.515
R²:   -0.099


## Going further

What would happen if you reversed it: train on Nairobi and test on Kampala? Try it in the cell below and compare the two errors.

In Session 3, you will see how to automate this kind of comparison across more than two cities at once with `GroupKFold` — see [section 4.2.3](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#423-groupkfold-cross-validation) for a preview.

**Continue on the site:** [Build a First Pipeline](https://hub.imt-atlantique.fr/datascience-toolkit/session2/practical_2/)